# Práctica 3 - Aprendizaje Supervisado I: Clasificación

**Materia:** Introducción al Aprendizaje Automático  
**Tema:** Aplicacion de fundamentos matematicos, probabilidad y estadistica a traves de un piplenie de AA

**Nombre y apellido:**  
**Fecha de entrega:**  Miércoles 22/04/2026, 23:59

---

## Objetivos

Este cuadernillo tiene como objetivo aplicar y comparar distintos modelos de clasificación sobre el dataset Iris, integrando conceptos vistos en clase:

- exploración de datos,
- partición en entrenamiento / validación / test,
- estandarización,
- regresión logística multinomial con TensorFlow usando mini-batch gradient descent,
- ajuste de hiperparámetros,
- comparación con kNN, Random Forest, SVM y Gradient Boosting,
- evaluación con métricas y tiempos de entrenamiento.


## Instrucciones

1. Hacé una copia personal de este cuaderno antes de editarlo.
2. Completá todas las celdas pedidas.
3. No borres las celdas de enunciado.
4. Ejecutá el cuaderno de principio a fin antes de entregar.
5. Dejá visibles los resultados y gráficos.
6. Entregá el archivo `.ipynb` con el nombre indicado por la cátedra.


## Parte 0 - Cargar librerías

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

import tensorflow as tf
from tensorflow import keras

# Para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

# Parte 1 — Exploración del dataset

Vamos a usar el dataset **Iris** disponible en `scikit-learn`.

Investigá y escribí brevemente de qué se trata este dataset en la página oficial de `scikit-learn`.

[Podés encontrarlo acá](https://scikit-learn.org/1.4/auto_examples/datasets/plot_iris_dataset.html)



_Reemplazá este texto por tu respuesta._

## Ejercicio 1.1
1. Cargar el dataset.
2. Construir un `DataFrame`.
3. Explorar:
   - cantidad de muestras,
   - cantidad de variables,
   - nombres de features (caracteristicas),
   - clases (etiquetas),
   - estadísticas descriptivas,
   - balance de clases.


In [ ]:
#Completá el código (...) y agregá lo que necesites

iris = load_iris()

X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

df = pd.DataFrame(..., columns=feature_names)
df["target"] = y
df["species"] = df["target"].map({i: name for i, name in enumerate(target_names)})

df.head()

df.describe(...)

df[...].value_counts()


## Visualización exploratoria

En esta parte exploramos los datos mediante plots (figuras)

## Pregunta 1.1

¿Qué variables parecen separar mejor las clases?

In [ ]:
#Completa el código explorando lo que necesités y respondé abajo tus observaciones
#Sugerencia: Hacé algunos gráficos simples como histogramas para mirar distribuciones
#y relaciones entre variables.

_Reemplazá este texto por tu respuesta._

# Parte 2 — División en entrenamiento, validación y test

Vamos a separar los datos en tres subconjuntos:

- **train**: para entrenar el modelo,
- **validation**: para comparar hiperparámetros,
- **test**: para la evaluación final.



In [ ]:
#Primero, dividimos del conjunto de entrenamiento el conjunto de test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

#segundo, del conjunto de entrenamiento, sacamos el conjunto de validación
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.25,   # 0.25 del 80% = 20% total
    random_state=42,
    stratify=y_train_full
)

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

### Ejercicio 2.1
 En este cuadernillo vamos a usar:

- 80% entrenamiento
- 10% validación
- 10% test

Completá los `...` en el código de abajo. Ojo con la proporción del `test_side` en validación

In [ ]:
#Primero, dividimos del conjunto de entrenamiento el conjunto de test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=...,
    random_state=42,
    stratify=y
)

#segundo, del conjunto de entrenamiento, sacamos el conjunto de validación
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=...,
    random_state=42,
    stratify=y_train_full
)

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

# Parte 3 — Estandarización

Muchos modelos son sensibles a la escala de las variables, especialmente:

- regresión logística,
- kNN,
- SVM.

### Muy importante
El `scaler` se ajusta solo con los datos de entrenamiento. Notá dónde aplicamos `fit_transform` y dónde `transform`.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Media aproximada en train escalado:", X_train_scaled.mean(axis=0))
print("Desvío aproximado en train escalado:", X_train_scaled.std(axis=0))

### Pregunta 3.1

 ¿Por qué no podemos usar todo el dataset para estandarizar (solo usamos el ajuste de train y con eso estandarizamos test y val)?


_Reemplazá este texto por tu respuesta._

# Parte 4 — Regresión logística multinomial con TensorFlow

Ahora vamos a entrenar un clasificador multiclase usando:

- capa densa con `softmax`,
- función de costo `sparse_categorical_crossentropy`,
- optimizador **SGD**,
- **mini-batch gradient descent** controlado por `batch_size`.

TensorFlow ya implementa MBGD internamente.

En el cuadernillo anterior vimos cada paso separado:



*   `model = ...`
*   `model.compile()`

pero podemos definir una función `modelo_log_multimod()` ue tiene como argumentos los hiperparámetros (en este caso, la tasa de aprendizaje).


In [ ]:
def modelo_log_multimod(learning_rate=0.01):
    model = keras.Sequential([
        keras.layers.Input(shape=(X_train_scaled.shape[1],)),
        keras.layers.Dense(3, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.SGD(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model



## Primer entrenamiento de prueba

Corré un entrenamiento simple para verificar que todo funciona (no toques ningun hiperparametro todavia).

In [ ]:
model = modelo_log_multimod(learning_rate=0.01)

history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=30,
    batch_size=16,
    verbose=1
)

history.history.keys()

In [ ]:
#visualizamos las Curvas de pérdida

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Curva de pérdida")
plt.legend()
plt.show()

In [ ]:
#Visualizamos como aumenta la exactitud en el entrenamiento
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Curva de accuracy")
plt.legend()
plt.show()

### Preguntas de chequeo de sanidad (informal)
- [ ] ¿La pérdida disminuye en ambos casos?
- [ ] ¿La exactitud mejora en promedio?
- [ ] ¿Train y validation se comportan parecido?

(Podes escribir una `x` entre los [ ])

# Parte 5 — Barrido de hiperparámetros en la logística multinomial

Vamos a variar tres hiperparámetros:

- `learning_rate`
- `epochs`
- `batch_size`

### Sugerencia
Usar estos valores (despues podés jugar con otros):
- `learning_rate`: `[0.001, 0.01, 0.1]`
- `epochs`: `[50, 80, 100]`
- `batch_size`: `[8, 16]`

Eso genera varias combinaciones.

In [ ]:
#Escribimos una grilla de busqueda de hiperparametros
param_grid_logreg = {
    "learning_rate": [0.001, 0.01, 0.1],
    "epochs": [50, 80, 100],
    "batch_size": [8, 16]
}

#En este caso, vamos a correr el modelo una cantidad de comb. totales:
print('Combinaciones totales:', len(list(ParameterGrid(param_grid_logreg))))

#Vemos algunas combinaciones
print('Ejemplos de combinaciones:')
list(ParameterGrid(param_grid_logreg))[:5]



In [ ]:
## Si, va a tardar bastante: como unos 5-7 minutos

logreg_results = []

for i, params in enumerate(ParameterGrid(param_grid_logreg)):
    print(f"Experimento {i+1}")
    model = modelo_log_multimod(learning_rate=params["learning_rate"])

    start = time.time()
    history = model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=params["epochs"],
        batch_size=params["batch_size"],
        verbose=0
    )
    end = time.time()

    #evaluamos el modelo en val despues que el modelo acaba el entrenamiento
    val_loss, val_acc = model.evaluate(X_val_scaled, y_val, verbose=0)

    print(f"Experimento {i+1} finalizado.")
    #guardamos los hiperparametros y cuanta perdida y exactitud nos dio
    #para train y val
    logreg_results.append({
        "learning_rate": params["learning_rate"],
        "epochs": params["epochs"],
        "batch_size": params["batch_size"],
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "train_time_sec": end - start,
        "history": history.history
    })

logreg_results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "history"} for r in logreg_results
])



In [ ]:
#miramos los resultados por aquellos que tienen exactitud en val mas alta
logreg_results_df.sort_values(by="val_accuracy", ascending=False).head(10)

## Ejercicios y preguntas

Con la tabla anterior, hacé el código que necesites abajo y respondé:

5.1. ¿Qué combinación dio mejor accuracy de validación?




In [ ]:
#Tu código aqui

_Reemplazá este texto por tu respuesta._


5.2. ¿Cuál fue la combinacion de parametros más lenta?


In [ ]:
#Tu código aqui

_Reemplazá este texto por tu respuesta._

5.3. ¿Se ve alguna combinación de hiperparametros 'mala'? (por ej., que de una exactitud de alrededor del 50%)

In [ ]:
#Tu código aqui

_Reemplazá este texto por tu respuesta._

## Visualización de resultados de la logística

Podés ordenar y comparar los experimentos con gráficos.

In [ ]:
# Top 10 configuraciones por accuracy de validación
top10 = logreg_results_df.sort_values(by="val_accuracy", ascending=False).head(10).copy()
top10["config"] = top10.apply(
    lambda row: f'lr={row["learning_rate"]}, ep={int(row["epochs"])}, bs={int(row["batch_size"])}',
    axis=1
)

plt.figure(figsize=(10, 5))
plt.bar(top10["config"], top10["val_accuracy"])
plt.xticks(rotation=70)
plt.ylabel("Validation accuracy")
plt.title("Top 10 configuraciones - Regresión logística multinomial")
plt.tight_layout()
plt.show()

In [ ]:
#y ver cuanto tardaron en correr ese top 10
plt.figure(figsize=(10, 5))
plt.bar(top10["config"], top10["train_time_sec"])
plt.xticks(rotation=70)
plt.ylabel("Tiempo de entrenamiento (s)")
plt.title("Tiempo de entrenamiento - Top 10 configuraciones")
plt.tight_layout()
plt.show()

##Mirando el gráfico anterior de exactitudes (eje x)
5.4. ¿Qué tasa de aprendizaje aparece en las mejores configuraciones?


_Reemplazá este texto por tu respuesta._

5.5. ¿Concluirías que influye el batch size?



_Reemplazá este texto por tu respuesta._

5.6. ¿Concluirías que más epochs siempre ayudaron a mejorar la exactitud en la validación?

_Reemplazá este texto por tu respuesta._

### Importante!

La elección de los hiperparámetros es crítica para el desempeño de un modelo.

No explorar ni ajustar los hiperparámetros puede llevar a resultados subóptimos, incluso cuando el modelo utilizado es adecuado para la tarea.

En la práctica, una buena selección de hiperparámetros puede marcar una diferencia significativa en la calidad de las predicciones.

## Curvas de aprendizaje de la mejor configuración

Vamos a seleccionar la mejor combinación según `val_accuracy` y graficá sus curvas.

In [ ]:
best_idx = logreg_results_df["val_accuracy"].idxmax()
best_logreg = logreg_results[best_idx]

best_logreg_params = {
    "learning_rate": best_logreg["learning_rate"],
    "epochs": best_logreg["epochs"],
    "batch_size": best_logreg["batch_size"]
}

best_logreg_params

In [ ]:
best_history = best_logreg["history"]

plt.figure(figsize=(7, 4))
plt.plot(best_history["loss"], label="train loss")
plt.plot(best_history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Mejor configuración: pérdida")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(best_history["accuracy"], label="train acc")
plt.plot(best_history["val_accuracy"], label="val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Mejor configuración: accuracy")
plt.legend()
plt.show()

### Pregunta 5.7.

Observá si la pérdida de validación es, en promedio, mayor o menor que la de entrenamiento. Hacé lo mismo para la exactitud.

¿Qué significa este comportamiento desde un punto de vista conceptual?

_Reemplazá este texto por tu respuesta._

# Parte 6 — Evaluación final de la mejor logística en test

Ahora sí: con la mejor configuración elegida usando **validación**, evaluamos en **test**.

In [ ]:
final_logreg_model = modelo_log_multimod(learning_rate=best_logreg_params["learning_rate"])

start = time.time()
final_logreg_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=best_logreg_params["epochs"],
    batch_size=best_logreg_params["batch_size"],
    verbose=0
)
end = time.time()

logreg_test_time = end - start

y_pred_proba = final_logreg_model.predict(X_test_scaled, verbose=0)
y_pred_logreg = np.argmax(y_pred_proba, axis=1)

logreg_metrics = {
    "model": "Logistic Regression Multinomial",
    "accuracy": accuracy_score(y_test, y_pred_logreg),
    "precision_macro": precision_score(y_test, y_pred_logreg, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_logreg, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_logreg, average="macro", zero_division=0),
    "train_time_sec": logreg_test_time
}

logreg_metrics

In [ ]:
print(classification_report(y_test, y_pred_logreg, target_names=target_names))

##Pregunta 6.1. (posgrado solamente)

Hacé un análisis corto de lo que dieron cada una de las métricas de la celda anterior, resultado de `classification_report()`

_Reemplazá este texto por tu respuesta._

# Parte 7 — Comparación con otros modelos

Ahora vamos a entrenar otros clasificadores y comparar sus resultados.

## Modelos
- kNN
- Random Forest
- Gradient Boosting
- SVM


## Importante
- Para **kNN** y **SVM**, usar datos escalados.
- Para **Random Forest** y **Gradient Boosting**, puede usarse el dataset original o escalado. En este cuadernillo usaremos el original.

## Funciones auxiliares para evaluar modelos

Vamos a definir algunas funciones para poder guardar modelos y resultados para despues poder comparar.

In [ ]:

def compute_metrics(y_true, y_pred, model_name, train_time):
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "train_time_sec": train_time
    }

def evaluate_sklearn_model(model, Xtr, ytr, Xte, yte, model_name):
    start = time.time()
    model.fit(Xtr, ytr)
    end = time.time()

    y_pred = model.predict(Xte)
    metrics = compute_metrics(yte, y_pred, model_name, end - start)
    return metrics, y_pred

## 7.1 kNN

Variar el hiperparámetro principal:
- `n_neighbors` = k en la teoria, numero de vecinos.



###Ejercicio 7.1.1

Elegi 5 valores para k y escribilos abajo en la lista `[...]`. Usamos el conjunto de validacion para elegir el k de mejor desempenio.


In [ ]:
knn_results = []

#Completa los 5 hiperparametros k que elegiste
for k in [...]:
    model = KNeighborsClassifier(n_neighbors=k)
    metrics, y_pred = evaluate_sklearn_model(
        model, X_train_scaled, y_train, X_val_scaled, y_val,
        model_name=f"kNN (k={k})"
    )
    metrics["n_neighbors"] = k
    knn_results.append(metrics)

pd.DataFrame(knn_results).sort_values(by="accuracy", ascending=False)

In [ ]:
#Esto es para fijarte cual de tus k tuvo el mejor desempenio
best_knn_row = pd.DataFrame(knn_results).sort_values(by="accuracy", ascending=False).iloc[0]
best_k = int(best_knn_row["n_neighbors"])

#Evaluamos en test con el mejor k
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn_metrics, y_pred_knn = evaluate_sklearn_model(
    best_knn, X_train_scaled, y_train, X_test_scaled, y_test,
    model_name=f"kNN (k={best_k})"
)

best_knn_metrics

## 7.2 Random Forest

Variar hiperparámetros principales:
- `n_estimators` : $T$ en teoría
- `max_depth` :máximo número de decisiones (splits) encadenadas desde la raíz hasta una hoja

###Ejercicio 7.2.1

Elegí 1 valor más para `n_estimators` y escribilos abajo en la lista en lugar de  `...`. Hacé lo mismo con `max_depth`.


In [ ]:
rf_results = []

for n_estimators in [50, ..., 200]:
    for max_depth in [None, 3, ..., 10]:
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42
        )
        metrics, y_pred = evaluate_sklearn_model(
            model, X_train, y_train, X_val, y_val,
            model_name=f"RF (n={n_estimators}, depth={max_depth})"
        )
        metrics["n_estimators"] = n_estimators
        metrics["max_depth"] = max_depth
        rf_results.append(metrics)

pd.DataFrame(rf_results).sort_values(by="accuracy", ascending=False).head(10)

In [ ]:

#Nos fijamos cual de los hiperparametros tuvo el mejor desempeño
best_rf_row = pd.DataFrame(rf_results).sort_values(by="accuracy", ascending=False).iloc[0]

best_rf = RandomForestClassifier(
    n_estimators=int(best_rf_row["n_estimators"]),
    max_depth=None if pd.isna(best_rf_row["max_depth"]) else int(best_rf_row["max_depth"]),
    random_state=42
)

#Lo usamos para test
best_rf_metrics, y_pred_rf = evaluate_sklearn_model(
    best_rf, X_train, y_train, X_test, y_test,
    model_name=f"Random Forest"
)

best_rf_metrics

## 7.3 Gradient Boosting

Variar:
- `n_estimators`
- `learning_rate` : tasa de aprendizaje
- `max_depth`

En `GradientBoostingClassifier`, la profundidad del árbol base se controla con `max_depth` dentro de un estimador de árbol. Para no complicar demasiado, vamos a variar `n_estimators` y `learning_rate`.

Si querés subir el nivel, podés investigar más hiperparámetros.

En este caso, te doy ya los hiperparámetros.

In [ ]:
gb_results = []

for n_estimators in [50, 100, 200]:
    for learning_rate in [0.01, 0.1, 0.2]:
        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            random_state=42
        )
        metrics, y_pred = evaluate_sklearn_model(
            model, X_train, y_train, X_val, y_val,
            model_name=f"GB (n={n_estimators}, lr={learning_rate})"
        )
        metrics["n_estimators"] = n_estimators
        metrics["learning_rate"] = learning_rate
        gb_results.append(metrics)

pd.DataFrame(gb_results).sort_values(by="accuracy", ascending=False).head(10)

In [ ]:
#Nos fijamos cual de los hiperparametros tuvo el mejor desempeño
best_gb_row = pd.DataFrame(gb_results).sort_values(by="accuracy", ascending=False).iloc[0]

best_gb = GradientBoostingClassifier(
    n_estimators=int(best_gb_row["n_estimators"]),
    learning_rate=float(best_gb_row["learning_rate"]),
    random_state=42
)

#Lo usamos para test
best_gb_metrics, y_pred_gb = evaluate_sklearn_model(
    best_gb, X_train, y_train, X_test, y_test,
    model_name="Gradient Boosting"
)

best_gb_metrics

## 7.4 SVM

Variar:
- `C`: controla cuánto castigo a los puntos que quedan mal ubicados luego de decidir la funcion $f_c(x)$
- `kernel`: si el problema es linealmente separable, usamos `linear`. Si no, usamos `rbf`.


En este caso, te doy ya los hiperparámetros.

In [ ]:
svm_results = []

for C in [0.1, 1, 10]:
    for kernel in ["linear", "rbf"]:
        model = SVC(C=C, kernel=kernel)
        metrics, y_pred = evaluate_sklearn_model(
            model, X_train_scaled, y_train, X_val_scaled, y_val,
            model_name=f"SVM (C={C}, kernel={kernel})"
        )
        metrics["C"] = C
        metrics["kernel"] = kernel
        svm_results.append(metrics)

pd.DataFrame(svm_results).sort_values(by="accuracy", ascending=False)

In [ ]:

#Nos fijamos cual de los hiperparametros tuvo el mejor desempeño
best_svm_row = pd.DataFrame(svm_results).sort_values(by="accuracy", ascending=False).iloc[0]

best_svm = SVC(
    C=float(best_svm_row["C"]),
    kernel=best_svm_row["kernel"]
)

#Lo usamos para test
best_svm_metrics, y_pred_svm = evaluate_sklearn_model(
    best_svm, X_train_scaled, y_train, X_test_scaled, y_test,
    model_name="SVM"
)

best_svm_metrics

# Parte 8 — Comparación final de modelos en test

Ahora juntamos todos los mejores modelos y los comparamos en test.

In [ ]:
final_results = pd.DataFrame([
    logreg_metrics,
    best_knn_metrics,
    best_rf_metrics,
    best_gb_metrics,
    best_svm_metrics
])

final_results.sort_values(by="accuracy", ascending=False)

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(final_results["model"], final_results["accuracy"])
plt.xticks(rotation=20)
plt.ylabel("Accuracy")
plt.title("Comparación de accuracy en test")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(final_results["model"], final_results["f1_macro"])
plt.xticks(rotation=20)
plt.ylabel("F1 macro")
plt.title("Comparación de F1 macro en test")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(final_results["model"], final_results["train_time_sec"])
plt.xticks(rotation=20)
plt.ylabel("Tiempo de entrenamiento (s)")
plt.title("Comparación de tiempo de entrenamiento")
plt.tight_layout()
plt.show()

## Matrices de confusión de los mejores modelos

Podés comparar visualmente dónde se equivoca cada modelo.

In [ ]:
predictions_dict = {
    "Logistic Regression (TF)": y_pred_logreg,
    f"kNN (k={best_k})": y_pred_knn,
    "Random Forest": y_pred_rf,
    "SVM": y_pred_svm,
    "Gradient Boosting": y_pred_gb
}

for model_name, y_pred in predictions_dict.items():
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    disp.plot(cmap="Blues")
    plt.title(f"Matriz de confusión - {model_name}")
    plt.show()

# Parte 9 — Análisis final

Respondé brevemente:

9.1. ¿Qué modelo obtuvo mejor accuracy?



_Reemplazá este texto por tu respuesta._

9.2. ¿Qué modelo obtuvo mejor F1 macro?



_Reemplazá este texto por tu respuesta._

9.3. ¿Cuál fue el más rápido de entrenar?


_Reemplazá este texto por tu respuesta._

9.4. Si tuvieras que elegir un modelo para este problema, ¿cuál usarías y por qué?

_Reemplazá este texto por tu respuesta._

# Parte 10 — Opcional

Si terminaste antes o querés explorar más, podés probar alguna de estas extensiones:

- comparar **con scaling vs sin scaling** en kNN o SVM,
- probar otras combinaciones de hiperparámetros,
- investigar diferencias entre `GradientBoostingClassifier` y `XGBoost`,


## Entrega

Antes de entregar, verificá lo siguiente:

- [ ] completé todas las consignas;
- [ ] ejecuté el cuaderno de principio a fin;
- [ ] dejé los resultados visibles;
- [ ] guardé el archivo con el nombre correcto.

Fin de la práctica.